### Stacking (Stacked Generalization)

**Core idea**  
Stacking improves over voting ensembles by **learning how to combine model predictions** instead of simply averaging them.

---

**From Voting → Stacking**

- Voting (regression):
  - Train multiple models
  - Take average of predictions  
  - All models are treated equally  

- Stacking:
  - Train multiple base models  
  - Use their predictions as **features**  
  - Train a **meta-model** to learn the best combination  

---

**Intuition**

Different models capture different patterns.  
Instead of assuming all models are equally good (like averaging), stacking lets another model learn:

> “Which model should I trust more for a given situation?”

---

**Step-by-step workflow**

Example dataset:

`cgpa | iq | package`

1. Train base models:
   - Linear Regression (LR)  
   - Decision Tree (DT)  
   - KNN  

2. Generate predictions from base models:

`cgpa | iq | package | lr_pred | dt_pred | knn_pred`

3. Create meta-dataset:

Inputs:
`lr_pred | dt_pred | knn_pred`  
Output:
`package`

4. Train meta-model:
   - Example: Random Forest Regressor (RFR)

5. Prediction phase:
   - New data → pass to all base models  
   - Collect predictions  
   - Pass predictions to meta-model  
   - Final output = meta-model prediction  

---

**Key insight**

Stacking =  
Base models → generate predictions → meta-model learns optimal combination  

---

**Difference from Bagging and Boosting**

- **Bagging**
  - Same model type (e.g., trees)
  - Trained independently on random samples
  - Output combined by averaging

- **Boosting**
  - Models trained sequentially
  - Each model focuses on previous errors
  - Strong dependency between models

- **Stacking**
  - Can use **different types of models**
  - Models trained independently
  - Final prediction comes from a **separate meta-model**, not direct averaging  

---

**Important note**

- Base model predictions are not directly used as final output  
- They are used as **input features for another model**

---

**Common mistake (critical)**

Training meta-model on predictions from same training data → **data leakage**

Correct approach:
- Use **cross-validation predictions (out-of-fold)**

---

**One-line revision**

> Stacking combines multiple models by training a meta-model on their predictions instead of averaging them.

# Problem 

### Overfitting in Stacking + Solution (Blending & K-Fold Stacking)

**Problem**  
In naive stacking:
- Base models are trained on full data  
- Predictions for meta-model are also generated on same data  

> This leads to **data leakage**  
> Especially worse when base models are high-capacity (DT, KNN)  
> Result: **overfitting**

---

**Solutions**

1. **Hold-out Method → Blending**  
2. **K-Fold Method → Proper Stacking**

---

### Blending (Hold-out Approach)

**Core idea**  
Separate data for:
- Training base models  
- Training meta-model  

---

**Dataset**

`cgpa | iq | package` (1000 rows)

---

**Step-by-step workflow**

1. Split data:

- `D → D_train (80%) + D_test (20%)`  
- `D_train → D_train_2 (60%) + D_validation (20%)`

---

2. Train base models on `D_train_2`:

- LR (m1)  
- DT (m2)  
- KNN (m3)  

---

3. Generate predictions on `D_validation`:

`lr_pred | dt_pred | knn_pred | package`  
Shape: `(200, 4)`

---

4. Train meta-model (RFR):

Inputs:
`lr_pred | dt_pred | knn_pred`  
Output:
`package`

---

5. Final evaluation:

- Use `D_test`  
- Pass through base models → meta-model  
- Compute metric (e.g., R² score)

---

**Key idea**

- Base models **never see validation data during training**  
- Meta-model is trained on **unseen predictions**  

>  This removes leakage → reduces overfitting  

---



### K-Fold Stacking (Proper Stacking using Cross-Validation)

**Core idea**  
Use K-Fold cross-validation to generate **out-of-fold predictions** for training the meta-model, avoiding data leakage and improving generalization.

---

**Dataset**

`cgpa | iq | package` (1000 rows)

Split:
- `D → D_train (80%) + D_test (20%)`

---

**Step 1: Generate meta-features using K-Fold (k = 4)**

- Split `D_train (800 rows)` into 4 folds (each 200 rows)

Base models:
- Linear Regression (LR)  
- Decision Tree (DT)  
- KNN  

---

**For each base model (example: LR)**

Perform 4 iterations:

1. Train on folds (1,2,3) → predict on fold 4 → 200 preds  
2. Train on folds (1,2,4) → predict on fold 3 → 200 preds  
3. Train on folds (1,3,4) → predict on fold 2 → 200 preds  
4. Train on folds (2,3,4) → predict on fold 1 → 200 preds  

> After 4 iterations:
- `lr_pred` = 800 values (out-of-fold predictions)

---

**Repeat same for:**
- DT → `dt_pred (800)`  
- KNN → `knn_pred (800)`

---

**New meta-dataset**

`lr_pred | dt_pred | knn_pred | package`  
Shape: `(800, 4)`

---

**Important observation**

- Each base model is trained **k times (here 4)**  
- Total base trainings = `3 models × 4 folds = 12 trainings`  
- These are **not final base models**

---

**Step 2: Train meta-model**

- Model: Random Forest Regressor (RFR)

Inputs:
`lr_pred | dt_pred | knn_pred`  
Output:
`package`

> Now meta-model is trained on **unseen predictions**

---

**Step 3: Train final base models**

- Train LR, DT, KNN again on full `D_train (800 rows)`  
- These become the **final base models** used in prediction  

---

**Step 4: Prediction phase**

- Input: `D_test`  
- Flow:
  - Pass data → base models → get predictions  
  - Pass predictions → meta-model  
  - Get final output  

- Evaluate using metric (e.g., R² score)

---

**Key insight**

> K-Fold stacking ensures every data point is predicted by a model that has **not seen it during training**

---

**Why this works better than blending**

- Uses full dataset efficiently  
- Reduces bias (no fixed hold-out split)  
- More robust meta-features  

---

**Common confusion**

- Models trained during K-Fold → ❌ not used for final prediction  
- Only used to **generate training data for meta-model**  

---

**One-line revision**

> K-Fold stacking trains the meta-model using out-of-fold predictions to eliminate leakage and improve generalization.

---
---

### Multi-layer Stacking (Deep Stacking Architecture)

**Core idea**  
Stacking is not limited to one meta-model.  
You can build multiple layers of models, where each layer learns from the previous one.

---

**Architecture**

- Layer 1: m1, m2, m3 (base models)  
- Layer 2: m4, m5, m6 (intermediate models)  
- Layer 3: m7 (final meta-model)  

Layers are fully connected (each layer uses outputs from previous layer)

---

### What you described = Blending (Hold-out based)

**Why?**
- Fixed splits (DT1, DT2, DT3)  
- Models trained on one split and predicted on another  
- No cross-validation  

This is multi-layer blending.

---

### How to convert this into proper Stacking

**Key change**  
Replace hold-out splits with K-Fold cross-validation at every layer.

---

### Step-by-step (Stacking version)

**Dataset**

D → D_train (900) + D_test (100)

---

### Layer 1 (m1, m2, m3)

- Apply K-Fold (e.g., k=3 or 5) on D_train  
- For each model:
  - Train on (k-1) folds  
  - Predict on remaining fold  

Generate out-of-fold predictions:

m1_pred | m2_pred | m3_pred (900 rows)

---

### Layer 2 (m4, m5, m6)

- Use Layer 1 predictions as input:

m1_pred | m2_pred | m3_pred → package  

- Again apply K-Fold  

Generate new out-of-fold predictions:

m4_pred | m5_pred | m6_pred (900 rows)

---

### Layer 3 (Final meta-model m7)

- Train on:

m4_pred | m5_pred | m6_pred → package  

(No need for K-Fold here if it is the final layer)

---

### Final Base Models (Important)

After building meta-features:

- Retrain:
  - m1, m2, m3 on full D_train  
  - Then m4, m5, m6 using full predictions from Layer 1  

These become final pipeline models.

---

### Prediction Flow

For D_test:

1. Pass through Layer 1 → get predictions  
2. Pass to Layer 2 → get predictions  
3. Pass to Layer 3 (m7) → final output  

---

### Key insight

Each layer must learn from out-of-fold predictions, not same-data predictions.

---

### Blending vs Stacking (multi-layer)

- Blending:
  - Uses fixed splits (DT1, DT2, DT3)  
  - Easier but wastes data  

- Stacking:
  - Uses K-Fold at every layer  
  - More complex but fully utilizes data  

---

### Common mistake

Using predictions from same training data at any layer leads to compounded leakage.

---

### revision

Multi-layer stacking builds deep model architectures by passing out-of-fold predictions across layers using K-Fold validation.

----
----

[winning kaggle model architecture](https://blogs.sas.com/content/subconsciousmusings/2017/05/18/stacked-ensemble-models-win-data-science-competitions/)

[sklearnStackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)

### StackingClassifier (sklearn)

**Core idea**  
Stack multiple base classifiers and train a final classifier on their predictions.  
Base models learn patterns → final model learns how to combine them.

---

### Important behavior

- Base estimators → trained on full training data  
- Final estimator → trained on **cross-validated predictions** (using `cross_val_predict`)  
- This avoids data leakage and improves generalization

---

### Key hyperparameters

**estimators**
- List of `(name, model)` tuples  
- Example: `[('lr', LR), ('dt', DT)]`  
- You can drop any model using `'drop'`  

---

**final_estimator**
- Model that combines base predictions  
- Default: `LogisticRegression`  
- Can be any classifier  

---

**cv**
- Controls how meta-features are generated  
- Options:
  - `None` → default 5-fold  
  - `int` → number of folds  
  - CV object  
  - `"prefit"` → assumes base models already trained  

Important:
- `"prefit"` → no cross-validation → high overfitting risk  
- CV is used for **prediction generation**, not evaluation  

---

**stack_method**
- How predictions are taken from base models  
- Options:
  - `'auto'` (default): tries  
    `predict_proba → decision_function → predict`
  - Or manually specify one  

Note:
- In binary classification with `predict_proba`, first column is dropped (to avoid collinearity)

---

**passthrough**
- `False` → only base model predictions used  
- `True` → original features + predictions used  

Insight:
- `True` can improve performance but may increase overfitting  

---

**n_jobs**
- Parallel training  
- `-1` → use all cores  

---

**verbose**
- Controls logging output  

---

### Attributes (important)

- `estimators_` → fitted base models  
- `final_estimator_` → trained meta-model  
- `named_estimators_` → access models by name  
- `stack_method_` → method used per estimator  

---

### Prediction flow

1. Input → base models  
2. Collect predictions  
3. Pass to final_estimator  
4. Output final prediction  

---

### Key insight

> StackingClassifier internally handles proper stacking using cross-validation, so you don’t manually need to generate out-of-fold predictions.

---

### Common mistakes

- Using `"prefit"` without care → leads to leakage  
- Using weakly diverse models → reduces stacking benefit  
- Ignoring `stack_method` mismatch (some models don’t support `predict_proba`)  

---

### When to use

- When multiple models perform differently  
- When you want to extract maximum performance  
- Common in competitions and high-performance systems  

---

### One-line revision

StackingClassifier trains a meta-model on cross-validated predictions of base models to optimally combine them.

# Example 

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [7]:
df = pd.read_csv("heart.csv")

df.sample(12)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
64,58,1,2,140,211,1,0,165,0,0.0,2,0,2,1
267,49,1,2,118,149,0,0,126,0,0.8,2,3,2,0
79,58,1,2,105,240,0,0,154,1,0.6,1,0,3,1
150,66,1,0,160,228,0,0,138,0,2.3,2,0,1,1
119,46,0,0,138,243,0,0,152,1,0.0,1,0,2,1
138,57,1,0,110,201,0,1,126,1,1.5,1,0,1,1
154,39,0,2,138,220,0,1,152,0,0.0,1,0,2,1
253,67,1,0,100,299,0,0,125,1,0.9,1,2,2,0
293,67,1,2,152,212,0,0,150,0,0.8,1,0,3,0
233,64,1,0,120,246,0,0,96,1,2.2,0,1,2,0


In [9]:
X = df.drop("target", axis=1)  
y = df["target"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
base_models = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('dt', DecisionTreeClassifier()),
    ('knn', KNeighborsClassifier())
]

In [12]:
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=RandomForestClassifier(),
    cv=5,                  # important → avoids leakage
    passthrough=False,     # only predictions used
    n_jobs=-1
)

In [13]:
stack_model.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('lr', ...), ('dt', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",RandomForestClassifier()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse

In [14]:
y_pred = stack_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7540983606557377
